# 模型推理与滚动分位数信号

读取 `rank_output/factor_rank_20日` 中的 45 维特征，加载 `MODEL_PATH` 指定的模型，生成股票分数和三种平滑信号。当前参数单元选用 **PAIR50E（50 epoch）**，结果写入 **model2_output**。

如需生成 10 epoch 模型的结果，将模型路径设为 `PAIR`，并将输出根目录设为 `model_output`。模型和输出目录应成对选择；两个实验均使用 `PAIR` 和 `PAIR_rank` 作为数据子目录名。

2022 年末 19 天用于窗口预热。日期 T 的模型输入截至 T−1，因此 T 日模型分数可纳入当日的 5/10/20 日滚动窗口，并用于 T 日开盘调仓。

In [10]:
import pandas as pd
import torch
from collections import deque
from tqdm import tqdm

from TOOL.data import preload, date_path, save_scores, RANK_PATH, MODEL_PATH, MODEL_OUTPUT_PATH
from TOOL.Struct import MarketField
from TOOL.factor import ALPHA_LIST, averRank
from TOOL.model import load_model

In [11]:
from pathlib import Path
MODEL_PATH=Path('/home/wenjy/FactorTrans/WORKFILE/PAIR50E')
MODEL_OUTPUT_PATH=Path('/home/wenjy/FactorTrans/WORKFILE/model2_output')

In [12]:
WINDOW_LIST = [5, 10, 20]
CACHE_SIZE = max(WINDOW_LIST)
years_list = ['2023', '2024', '2025', '2026']
base_info, dates_list, dates_path_list = preload(['2022', *years_list])
dates_list = [date for date in dates_list if date[:4] == '2022'][-(CACHE_SIZE - 1):] + [
    date for date in dates_list if date[:4] in years_list
]
factors_path_list = [date_path(RANK_PATH, date) for date in dates_list]
factor_cache = deque(maxlen=CACHE_SIZE)

torch.set_num_threads(4)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_model(MODEL_PATH, device)
print('模型：', MODEL_PATH, '设备：', device)
print('含预热：', len(dates_list), dates_list[0], dates_list[-1])

模型： /home/wenjy/FactorTrans/WORKFILE/PAIR50E 设备： cuda
含预热： 927 2022-12-06 2026-09-30


## 一次推理生成四种信号

`PAIR` 子目录保存模型直接输出的 `model` 分数。`PAIR_rank` 子目录保存四列：`model` 为当日分数的横截面分位数；`rank_5`、`rank_10`、`rank_20` 为逐日分位数在对应窗口内取均值后再次排名的结果。

当日模型分数先加入缓存，再计算滚动窗口。窗口未满时相应列保存 NaN；2023 年正式回测开始时三个窗口均已完整。

In [14]:
for date, factor_path in tqdm(list(zip(dates_list, factors_path_list))):
    factor_info = pd.read_parquet(factor_path).set_index('order_book_id')
    x = factor_info[ALPHA_LIST].fillna(0.5)

    #=======
    with torch.inference_mode():
        pred = model(torch.as_tensor(x.to_numpy(), dtype=torch.float32, device=device))
    rank_today = pd.DataFrame({'model': pred.squeeze(-1).cpu().numpy()}, index=x.index)
    save_scores(rank_today, MODEL_OUTPUT_PATH / 'PAIR', date)

    factor_cache.append(rank_today.rename_axis('order_book_id').reset_index().assign(datetime=date))
    factor_summary = MarketField(pd.concat(factor_cache))
    rank_output = pd.DataFrame(index=factor_summary.assets)
    rank_output['model'] = rank_today['model'].rank(pct=True)
    for WINDOW in WINDOW_LIST:
        rank_output[f'rank_{WINDOW}'] = (
            averRank(factor_summary, 'model', WINDOW)
            if len(factor_cache) >= WINDOW else float('nan')
        )
    save_scores(rank_output, MODEL_OUTPUT_PATH / 'PAIR_rank', date)
    #=======

100%|██████████| 927/927 [01:29<00:00, 10.38it/s]


In [4]:
print('原始输出：', MODEL_OUTPUT_PATH / 'PAIR')
print('滚动排名：', MODEL_OUTPUT_PATH / 'PAIR_rank')
rank_output.head()

原始输出： /home/wenjy/FactorTrans/WORKFILE/model_output/PAIR
滚动排名： /home/wenjy/FactorTrans/WORKFILE/model_output/PAIR_rank


,model,rank_5,rank_10,rank_20
000001.XSHE,0.711277,0.689259,0.635267,0.637948
000002.XSHE,0.040973,0.134214,0.268045,0.294275
000006.XSHE,0.144936,0.277618,0.278576,0.329313
000007.XSHE,0.747080,0.598890,0.550450,0.479226
000008.XSHE,0.888953,0.924947,0.934520,0.924373
